# 04 - Gold Layer: Load Dimensional Model
Read from Silver zone (cleansed) and load the star schema (Gold zone).

**Medallion Architecture - Gold**: Business-level dimensional model (dims & facts).

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up schemas, paths, and runtime variables for the Food Inspection project.

Medallion layers are separated by schema for Delta tables and by folder path for Parquet snapshots.

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


All widgets cleared.


## Widget Parameters

raw_path    = /Volumes/workspace/food_inspection/raw_data
bronze_path = /Volumes/workspace/foodinspection_bronze/food_inspection
silver_path = /Volumes/workspace/foodinspection_silver/food_inspection
gold_path   = /Volumes/workspace/foodinspection_gold/food_inspection
bronze_schema = foodinspection_bronze
silver_schema = foodinspection_silver
gold_schema   = foodinspection_gold


...

Schemas ready: foodinspection_bronze, foodinspection_silver, foodinspection_gold


## Verify Raw Data Files Exist

Raw data files:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)
  backup_Food_Inspections_20260411.csv (330.70 MB)
  parquet_snapshots/ (0.00 MB)


## Helper Utilities

Configuration ready. Variables and snapshot helpers are available via %run ./00_setup_config


In [0]:
from pyspark.sql.functions import (
    col, lit, when, coalesce, trim, upper, concat, concat_ws, expr,
    monotonically_increasing_id, row_number, dense_rank,
    year, month, dayofmonth, dayofweek, quarter, date_format,
    min as spark_min, max as spark_max, current_date, current_timestamp
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

In [0]:
def get_etl_job_id():
    """Get the current notebook path, or 'interactive' if unavailable."""
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except Exception:
        return "interactive"

ETL_JOB_ID = get_etl_job_id()

def add_gold_lineage(df):
    """Add Gold layer lineage columns: created_at, updated_at, etl_job_id."""
    return (
        df
        .withColumn("created_at", current_timestamp())
        .withColumn("updated_at", current_timestamp())
        .withColumn("etl_job_id", lit(ETL_JOB_ID))
    )

# Reuse snapshot helper from setup notebook if available via %run

## 1. Load Silver Tables

In [0]:
df_chi_insp = spark.table(f"{SILVER_SCHEMA}.chicago_inspections")
df_dal_insp = spark.table(f"{SILVER_SCHEMA}.dallas_inspections")
df_chi_viol = spark.table(f"{SILVER_SCHEMA}.chicago_violations")
df_dal_viol = spark.table(f"{SILVER_SCHEMA}.dallas_violations")

print(f"Chicago inspections: {df_chi_insp.count()}")
print(f"Dallas inspections:  {df_dal_insp.count()}")
print(f"Chicago violations:  {df_chi_viol.count()}")
print(f"Dallas violations:   {df_dal_viol.count()}")

Chicago inspections: 221877
Dallas inspections:  54090
Chicago violations:  954917
Dallas violations:   308877


---
## 2. dim_date

In [0]:
# Get date range from both datasets
chi_dates = df_chi_insp.select(
    spark_min("Inspection_Date").alias("min_date"),
    spark_max("Inspection_Date").alias("max_date")
).collect()[0]

dal_dates = df_dal_insp.select(
    spark_min("Inspection_Date").alias("min_date"),
    spark_max("Inspection_Date").alias("max_date")
).collect()[0]

min_date = min(chi_dates["min_date"], dal_dates["min_date"])
max_date = max(chi_dates["max_date"], dal_dates["max_date"])

print(f"Date range: {min_date} to {max_date}")

Date range: 2010-01-04 to 2026-04-08


In [0]:
# Generate date dimension
df_dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{min_date}'),
            to_date('{max_date}'),
            interval 1 day
        )) AS full_date
    """)
    .withColumn("date_key", (year(col("full_date")) * 10000 + month(col("full_date")) * 100 + dayofmonth(col("full_date"))).cast(IntegerType()))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("year", year(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
)

df_dim_date = add_gold_lineage(df_dim_date)

(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD_SCHEMA}.dim_date")
)

write_parquet_snapshot(df_dim_date, GOLD_PATH, "dim_date")

print(f"dim_date: {df_dim_date.count()} rows")
display(df_dim_date.limit(10))

Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/dim_date/2026/04/13/run_id=20260413_182145
dim_date: 5939 rows


full_date,date_key,day,month,month_name,quarter,year,day_of_week,day_name,created_at,updated_at,etl_job_id
2010-01-04,20100104,4,1,January,1,2010,2,Monday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-05,20100105,5,1,January,1,2010,3,Tuesday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-06,20100106,6,1,January,1,2010,4,Wednesday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-07,20100107,7,1,January,1,2010,5,Thursday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-08,20100108,8,1,January,1,2010,6,Friday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-09,20100109,9,1,January,1,2010,7,Saturday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-10,20100110,10,1,January,1,2010,1,Sunday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-11,20100111,11,1,January,1,2010,2,Monday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-12,20100112,12,1,January,1,2010,3,Tuesday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2010-01-13,20100113,13,1,January,1,2010,4,Wednesday,2026-04-13T18:22:01.160Z,2026-04-13T18:22:01.160Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load


---
## 3. dim_restaurant

### 3.1 Build staging data (deduplicated per business key)
Business key: `restaurant_name + source_city + license_number`

Tracked SCD2 attributes: `facility_type`, `risk_category`, `aka_name`

In [0]:
# Chicago restaurants - deduplicate by business key, keep latest values
# Add Inspection_ID as tiebreaker for rows with same date
w_chi = Window.partitionBy("DBA_Name", "source_city", "License").orderBy(col("Inspection_Date").desc(), col("Inspection_ID").desc())
df_chi_restaurants = (
    df_chi_insp
    .withColumn("_rn", row_number().over(w_chi))
    .filter(col("_rn") == 1)
    .select(
        col("DBA_Name").alias("restaurant_name"),
        col("AKA_Name").alias("aka_name"),
        col("License").cast("string").alias("license_number"),
        col("Facility_Type").alias("facility_type"),
        col("Risk").alias("risk_category"),
        col("source_city")
    )
    .dropDuplicates(["restaurant_name", "source_city", "license_number"])  # safety net
)

# Dallas restaurants - deduplicate by business key
w_dal = Window.partitionBy("Restaurant_Name", "source_city").orderBy(col("Inspection_Date").desc(), col("Inspection_ID").desc())
df_dal_restaurants = (
    df_dal_insp
    .withColumn("_rn", row_number().over(w_dal))
    .filter(col("_rn") == 1)
    .select(
        col("Restaurant_Name").alias("restaurant_name"),
        lit(None).cast("string").alias("aka_name"),
        lit(None).cast("string").alias("license_number"),
        lit(None).cast("string").alias("facility_type"),
        lit(None).cast("string").alias("risk_category"),
        col("source_city")
    )
    .dropDuplicates(["restaurant_name", "source_city", "license_number"])  # safety net
)

# Union staging data
df_staging_restaurant = df_chi_restaurants.unionByName(df_dal_restaurants)

df_staging_restaurant.createOrReplaceTempView("staging_restaurant")
print(f"Staging restaurants: {df_staging_restaurant.count()} (Chicago: {df_chi_restaurants.count()}, Dallas: {df_dal_restaurants.count()})")

Staging restaurants: 45544 (Chicago: 36857, Dallas: 8687)


### 3.2 SCD Type 2 Merge on dim_restaurant
- First run: creates the table with initial load
- Subsequent runs: expires changed rows, inserts new current rows

In [0]:
table_name = f"{GOLD_SCHEMA}.dim_restaurant"

# Check if dim_restaurant exists as a Delta table
table_exists = False
try:
    if spark.catalog.tableExists(table_name):
        # Verify it's actually a Delta table
        table_format = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
        if table_format == "delta":
            target = DeltaTable.forName(spark, table_name)
            table_exists = True
            print("dim_restaurant exists as Delta — applying SCD Type 2 merge...")
        else:
            print(f"dim_restaurant exists but is {table_format} format — dropping and recreating as Delta...")
            spark.sql(f"DROP TABLE IF EXISTS {table_name}")
except Exception as e:
    print(f"Table check: {e} — will create fresh.")
    table_exists = False

if table_exists:

    # Step 1: Expire changed rows
    target.alias("t").merge(
        df_staging_restaurant.alias("s"),
        """
        t.restaurant_name = s.restaurant_name
        AND t.source_city = s.source_city
        AND COALESCE(t.license_number, '') = COALESCE(s.license_number, '')
        AND t.is_current = True
        """
    ).whenMatchedUpdate(
        condition="""
            COALESCE(t.facility_type, '') != COALESCE(s.facility_type, '')
            OR COALESCE(t.risk_category, '') != COALESCE(s.risk_category, '')
            OR COALESCE(t.aka_name, '') != COALESCE(s.aka_name, '')
        """,
        set={
            "effective_end_date": current_date(),
            "is_current": lit(False),
            "updated_at": current_timestamp()
        }
    ).execute()

    print("Step 1: Expired changed rows.")

    # Step 2: Insert new current rows for changed + brand new records
    new_or_changed = spark.sql(f"""
        SELECT s.*
        FROM staging_restaurant s
        LEFT JOIN {table_name} t
        ON s.restaurant_name = t.restaurant_name
           AND s.source_city = t.source_city
           AND COALESCE(s.license_number, '') = COALESCE(t.license_number, '')
           AND t.is_current = True
        WHERE t.restaurant_key IS NULL
    """)

    if new_or_changed.count() > 0:
        max_key = spark.sql(f"SELECT COALESCE(MAX(restaurant_key), 0) AS mk FROM {table_name}").collect()[0]["mk"]

        new_rows = (
            new_or_changed
            .withColumn("restaurant_key", monotonically_increasing_id() + max_key + 1)
            .withColumn("effective_start_date", current_date())
            .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
            .withColumn("is_current", lit(True))
        )
        new_rows = add_gold_lineage(new_rows)
        new_rows.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"Step 2: Inserted {new_rows.count()} new/updated rows.")
    else:
        print("Step 2: No new or changed records.")

else:
    print("dim_restaurant does not exist — creating initial load with APPEND mode...")

    df_dim_restaurant = (
        df_staging_restaurant
        .withColumn("restaurant_key", monotonically_increasing_id())
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
    )
    df_dim_restaurant = add_gold_lineage(df_dim_restaurant)

    # Use append mode — saveAsTable creates the Delta table if it doesn't exist
    # Never use overwrite for SCD2 tables to preserve history on re-runs
    (
        df_dim_restaurant.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )
    print(f"Initial load (append): {df_dim_restaurant.count()} rows")

dim_restaurant does not exist — creating initial load with APPEND mode...
Initial load (append): 45544 rows


In [0]:
# Show SCD2 state
display(spark.sql(f"""
    SELECT is_current, COUNT(*) AS row_count
    FROM {GOLD_SCHEMA}.dim_restaurant
    GROUP BY is_current
"""))

dim_restaurant = spark.table(f"{GOLD_SCHEMA}.dim_restaurant")
write_parquet_snapshot(dim_restaurant, GOLD_PATH, "dim_restaurant")
print(f"dim_restaurant total: {dim_restaurant.count()} rows")
display(dim_restaurant.filter(col("is_current") == True).limit(10))

is_current,row_count
true,45544


Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/dim_restaurant/2026/04/13/run_id=20260413_182145
dim_restaurant total: 45544 rows


restaurant_name,aka_name,license_number,facility_type,risk_category,source_city,restaurant_key,effective_start_date,effective_end_date,is_current,created_at,updated_at,etl_job_id
11 DINING,FOODA SWITCHING STATIONS,2569547,Restaurant,Risk 1 (High),Chicago,0,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
111 COFFEE BAR,111 COFFEE BAR,2608177,Restaurant,Risk 2 (Medium),Chicago,1,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
"1200 JES FUELS,INC",CLARK,2704091,Grocery Store,Risk 2 (Medium),Chicago,2,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
"14 W. HUBBARD, LLC",LAO EIGHTEEN,2109816,Restaurant,Risk 1 (High),Chicago,3,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
1ST FRUITS ACADEMY INC,null,2054460,Daycare (2 - 6 Years),Risk 1 (High),Chicago,4,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
3 ABEJAS CORP.,LAS 3 ABEJAS,2404593,Mobile frozen dessert vendor,Risk 3 (Low),Chicago,5,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
5826 DIVISION FOOD,MOE & ACE,2464280,Grocery Store,Risk 2 (Medium),Chicago,6,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
63RD ST SUPER MART,63RD ST SUPER MART,3021136,Restaurant,Risk 1 (High),Chicago,7,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
7 ELEVEN,7 ELEVEN,1248291,Grocery Store,Risk 2 (Medium),Chicago,8,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
7 ELEVEN #34724J,7 ELEVEN #34724J,2283772,Grocery Store,Risk 3 (Low),Chicago,9,2026-04-13,9999-12-31,true,2026-04-13T18:22:09.888Z,2026-04-13T18:22:09.888Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load


---
## 4. dim_location

In [0]:
# Chicago locations
df_chi_locations = (
    df_chi_insp.select(
        col("Address").alias("address"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Zip").alias("zip"),
        expr("try_cast(Latitude as double)").alias("latitude"),
        expr("try_cast(Longitude as double)").alias("longitude"),
        col("source_city")
    )
)

# Dallas locations
df_dal_locations = (
    df_dal_insp.select(
        col("Street_Address").alias("address"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Zip_Code").alias("zip"),
        expr("try_cast(Latitude as double)").alias("latitude"),
        expr("try_cast(Longitude as double)").alias("longitude"),
        col("source_city")
    )
)

df_dim_location = (
    df_chi_locations.unionByName(df_dal_locations)
    .groupBy("address", "city", "state", "zip", "source_city")
    .agg(
        spark_max("latitude").alias("latitude"),
        spark_max("longitude").alias("longitude")
    )
    .withColumn("location_key", monotonically_increasing_id())
)


df_dim_location = add_gold_lineage(df_dim_location)

(
    df_dim_location.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD_SCHEMA}.dim_location")
)

write_parquet_snapshot(df_dim_location, GOLD_PATH, "dim_location")

print(f"dim_location: {df_dim_location.count()} rows")
display(df_dim_location.limit(10))

Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/dim_location/2026/04/13/run_id=20260413_182145
dim_location: 25977 rows


address,city,state,zip,source_city,latitude,longitude,location_key,created_at,updated_at,etl_job_id
1538 N CLYBOURN AVE,CHICAGO,IL,60610,Chicago,41.90931377383,-87.64750167889,0,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
3802 W DIVERSEY AVE,CHICAGO,IL,60647,Chicago,41.93195943809,-87.72223430062,1,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
3250 W Monroe St (100S),CHICAGO,IL,60624,Chicago,41.87974117351,-87.70808198851,2,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
1308 S CENTRAL PARK AVE,CHICAGO,IL,60623,Chicago,41.86409878432,-87.71550171632,3,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
6156 W BELMONT AVE,CHICAGO,IL,60634,Chicago,41.93849726116,-87.78087838926,4,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2039-2041 W NORTH AVE,CHICAGO,IL,60647,Chicago,41.91034561902,-87.67906657744,5,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
3217 W NORTH AVE,CHICAGO,IL,60647,Chicago,41.90999732785,-87.70765776138,6,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
5211 W NORTH AVE,CHICAGO,IL,60639,Chicago,41.90938968043,-87.75594521721,7,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
500 W MADISON ST,CHICAGO,IL,60661,Chicago,41.88199433821,41.88199433821,8,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
6011 S STONY ISLAND AVE,null,IL,60637,Chicago,41.78574881275,-87.58643828073,9,2026-04-13T18:22:23.482Z,2026-04-13T18:22:23.482Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load


---
## 5. dim_inspection_type

In [0]:
# Distinct inspection types from both cities
df_chi_types = df_chi_insp.select(col("Inspection_Type").alias("inspection_type_name"), col("source_city")).distinct()
df_dal_types = df_dal_insp.select(col("Inspection_Type").alias("inspection_type_name"), col("source_city")).distinct()

df_dim_inspection_type = (
    df_chi_types.unionByName(df_dal_types)
    .distinct()
    .withColumn("inspection_type_key", monotonically_increasing_id())
)

df_dim_inspection_type = add_gold_lineage(df_dim_inspection_type)

(
    df_dim_inspection_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD_SCHEMA}.dim_inspection_type")
)

write_parquet_snapshot(df_dim_inspection_type, GOLD_PATH, "dim_inspection_type")

print(f"dim_inspection_type: {df_dim_inspection_type.count()} rows")
display(df_dim_inspection_type)

Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/dim_inspection_type/2026/04/13/run_id=20260413_182145
dim_inspection_type: 64 rows


inspection_type_name,source_city,inspection_type_key,created_at,updated_at,etl_job_id
SMOKING COMPLAINT,Chicago,0,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
CANVASS/SPECIAL EVENT,Chicago,1,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Non-Inspection,Chicago,2,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
CANVASS SPECIAL EVENTS,Chicago,3,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
SFP,Chicago,4,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
CITF,Chicago,5,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
TWO PEOPLE ATE AND GOT SICK.,Chicago,6,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
TASK FORCE NIGHT,Chicago,7,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Special Task Force,Chicago,8,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
SPECIAL TASK FORCE,Chicago,9,2026-04-13T18:22:31.069Z,2026-04-13T18:22:31.069Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load


---
## 6. dim_violation
Stores both Chicago and Dallas violation codes (they don't need to match).

In [0]:
# Chicago distinct violations
df_chi_viol_dim = (
    df_chi_viol.select(
        col("source_city"),
        col("violation_code"),
        col("violation_description"),
        lit(None).cast(IntegerType()).alias("violation_points")
    )
)

# Dallas distinct violations aggregated by business key
df_dal_viol_dim = (
    df_dal_viol.groupBy("source_city", "violation_code", "violation_description")
    .agg(spark_max("violation_points").alias("violation_points"))
)

# Union and assign surrogate keys
df_dim_violation = (
    df_chi_viol_dim.unionByName(df_dal_viol_dim)
    .dropDuplicates(["source_city", "violation_code", "violation_description"])
    .withColumn("violation_key", monotonically_increasing_id())
)

df_dim_violation = add_gold_lineage(df_dim_violation)

(
    df_dim_violation.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD_SCHEMA}.dim_violation")
)

write_parquet_snapshot(df_dim_violation, GOLD_PATH, "dim_violation")

print(f"dim_violation: {df_dim_violation.count()} rows")
display(df_dim_violation.limit(20))

Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/dim_violation/2026/04/13/run_id=20260413_182145
dim_violation: 15465 rows


source_city,violation_code,violation_description,violation_points,violation_key,created_at,updated_at,etl_job_id
Chicago,33,PROPER COOLING METHODS USED; ADEQUATE EQUIPMENT FOR TEMPERATURE CONTROL,null,0,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: 4-101.19 OBSERVED MILK CRATES BEING USED AS SHELVING THROUGHOUT THE PREP AND STORAGE AREAS. MUST REMOVE AND PROVIDE SHELVING THAT IS 6"""" OFF THE FLOOR AND PROVIDES FLOOR ACCESSIBILITY TO ENSURE ADEQUATE FACILITY CLEANING.""",null,1,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""49. NON-FOOD/FOOD CONTACT SURFACES CLEAN - Comments: 4-601.11(C) OBSERVED THE INSIDE OF THE DEEP FRYER (REAR) OIL FILTERING UNIT WITH A BUILD-UP OF GREASE AND FOOD DEBRIS. NOTED EXCESS FOOD DEBRIS ON ALL SURFACES OF HEAVY EQUIPMENT THROUGHOUT FRONT AND REAR PREP/DISHWASHING AREA. INSTRUCTED TO CLEAN TO PREVENT PEST HARBORAGE.",null,2,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: NOTED FRONT ENTRANCE DOOR TO THE BUILDING NOT RODENT PROOF WITH GAP AT THE BOTTOM OF DOOR (ABOUT 1/4 INCH). INSTRUCTED TO SEAL GAPS TO PREVENT ENTRY POINTS FOR PESTS.",null,3,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED 1/4"""" GAP ON BOTTOM RIGHT CORNER OF FRONT ENTRANCE DOOR. INSTRUCTED MANAGER TO SEAL ALL OUTER OPENINGS ON DOOR TO PREVENT PEST ENTRY.",null,4,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""36. THERMOMETERS PROVIDED & ACCURATE - Comments: OBSERVED NO AMBIENT AIR THERMOMETER INSIDE THE 1-DOOR REACH-IN COOLER LOCATED IN THE BASKIN ROBBIN AREA. INSTRUCTED TO PROVIDE CONSPICUOUS AND EASILY READABLE THERMOMETERS INSIDE ALL REFRIGERATOR UNITS TO MONITOR THE AMBIENT AIR TEMPERATURE OF EQUIPMENT PROPERLY. AN AIR AMIBIENT THERMOMETER WAS PROVIDED IN THE 1-DOOR COOLER DURING THE INSPECTION.",null,5,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED GAP GREATER THAN 1/4"""" BETWEEN THE REAR KITCHEN AREA EXTERIOR DOOR AND DOORFRAME ALONG THE BOTTOM EDGE. INSTRUCTED TO REPAIR OR REPLACE DOOR TO ELIMINATE GAP GREATER THAN 1/4"""" AND MAINTAIN ALL EXTERIOR DOORS AND OTHER OUTER OPENINGS FREE FROM GAPS TO PREVENT ENTRY OF INSECTS OR OTHER PESTS.",null,6,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,21,* CERTIFIED FOOD MANAGER ON SITE WHEN POTENTIALLY HAZARDOUS FOODS ARE PREPARED AND SERVED,null,7,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED - Comments: HIGH TEMP DISH MACHINE NOT REACHING 180F AT FINAL RINSE. UNIT IN QUESTION IN USE FOR BREAKFAST DISHES. TAGGED UNIT """"HELD FOR INSPECTION"""" DO NOT USE. INSTRUCTED TO REPAIR UNIT",null,8,2026-04-13T18:22:38.908Z,2026-04-13T18:22:38.908Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
Chicago,null,"""34. FLOORS: CONSTRUCTED PER CODE, CLEANED, GOOD REPAIR, COVING INSTALLED, DUST-LESS CLEANING METHODS USED - Comments: FLOORS NOT CLEAN BASEMENT AREA, UNDER AND BEHIND EQUIPMENT, INSTRUCTED TO CLEAN DAILY,",null,9,2026-04-13T18:22:38

---
## 7. fact_inspection

In [0]:
# Reload dims to get surrogate keys (filter dim_restaurant for current rows only)
dim_restaurant = spark.table(f"{GOLD_SCHEMA}.dim_restaurant").filter(col("is_current") == True)
dim_location = spark.table(f"{GOLD_SCHEMA}.dim_location")
dim_inspection_type = spark.table(f"{GOLD_SCHEMA}.dim_inspection_type")
dim_date = spark.table(f"{GOLD_SCHEMA}.dim_date")

### 7.1 Chicago Fact

In [0]:
df_chi_fact = (
    df_chi_insp
    .join(
        dim_restaurant.select("restaurant_key", "restaurant_name", "license_number", "source_city"),
        (df_chi_insp["DBA_Name"] == dim_restaurant["restaurant_name"]) &
        (df_chi_insp["License"].cast("string") == dim_restaurant["license_number"]) &
        (df_chi_insp["source_city"] == dim_restaurant["source_city"]),
        "left"
    )

    .join(
        dim_location,
        (df_chi_insp["Address"] == dim_location["address"]) &
        (df_chi_insp["City"] == dim_location["city"]) &
        (df_chi_insp["State"] == dim_location["state"]) &
        (df_chi_insp["Zip"] == dim_location["zip"]) &
        (df_chi_insp["source_city"] == dim_location["source_city"]),
        "left"
    )

    .join(
        dim_inspection_type,
        (df_chi_insp["Inspection_Type"] == dim_inspection_type["inspection_type_name"]) &
        (df_chi_insp["source_city"] == dim_inspection_type["source_city"]),
        "left"
    )
    .join(
        dim_date,
        df_chi_insp["Inspection_Date"] == dim_date["full_date"],
        "left"
    )
    .select(
        df_chi_insp["Inspection_ID"].alias("inspection_id"),
        col("restaurant_key"),
        col("location_key"),
        col("inspection_type_key"),
        col("date_key"),
        df_chi_insp["Results"].alias("inspection_result"),
        df_chi_insp["Inspection_Score"].alias("inspection_score"),
        df_chi_insp["source_city"]
    )
)

### 7.2 Dallas Fact

In [0]:
df_dal_fact = (
    df_dal_insp
    .join(
        dim_restaurant.filter(col("is_current") == True).select("restaurant_key", "restaurant_name", "source_city"),
        (df_dal_insp["Restaurant_Name"] == dim_restaurant["restaurant_name"]) &
        (df_dal_insp["source_city"] == dim_restaurant["source_city"]),
        "left"
    )
    
    .join(
        dim_location,
        (df_dal_insp["Street_Address"] == dim_location["address"]) &
        (df_dal_insp["City"] == dim_location["city"]) &
        (df_dal_insp["State"] == dim_location["state"]) &
        (df_dal_insp["Zip_Code"] == dim_location["zip"]) &
        (df_dal_insp["source_city"] == dim_location["source_city"]),
        "left"
    )

    .join(
        dim_inspection_type,
        (df_dal_insp["Inspection_Type"] == dim_inspection_type["inspection_type_name"]) &
        (df_dal_insp["source_city"] == dim_inspection_type["source_city"]),
        "left"
    )
    .join(
        dim_date,
        df_dal_insp["Inspection_Date"] == dim_date["full_date"],
        "left"
    )
    .select(
        df_dal_insp["Inspection_ID"].alias("inspection_id"),
        col("restaurant_key"),
        col("location_key"),
        col("inspection_type_key"),
        col("date_key"),
        # Derive inspection result from score for Dallas
        when(df_dal_insp["Inspection_Score"] >= 90, lit("Pass"))
        .when(df_dal_insp["Inspection_Score"] >= 80, lit("Pass w/ Conditions"))
        .when(df_dal_insp["Inspection_Score"] >= 70, lit("Fail"))
        .when(df_dal_insp["Inspection_Score"] < 70, lit("Fail"))
        .otherwise(lit(None).cast("string"))
        .alias("inspection_result"),
        df_dal_insp["Inspection_Score"].alias("inspection_score"),
        df_dal_insp["source_city"]
    )
)

### 7.3 Union & Write Fact Table

In [0]:
df_fact_inspection = (
    df_chi_fact.unionByName(df_dal_fact)
    .withColumn("inspection_key", monotonically_increasing_id())
)

df_fact_inspection = add_gold_lineage(df_fact_inspection)

(
    df_fact_inspection.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD_SCHEMA}.fact_inspection")
)

write_parquet_snapshot(df_fact_inspection, GOLD_PATH, "fact_inspection")

print(f"fact_inspection: {df_fact_inspection.count()} rows")
display(df_fact_inspection.limit(10))

Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/fact_inspection/2026/04/13/run_id=20260413_182145
fact_inspection: 275967 rows


inspection_id,restaurant_key,location_key,inspection_type_key,date_key,inspection_result,inspection_score,source_city,inspection_key,created_at,updated_at,etl_job_id
2633665,26881,8482,15,20260408,Pass,90,Chicago,0,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633654,26752,9156,29,20260408,Pass,90,Chicago,1,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633653,27816,4255,35,20260408,Pass,90,Chicago,2,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633638,18532,14749,49,20260408,Fail,70,Chicago,3,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633566,29804,0,15,20260407,Pass,90,Chicago,4,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633540,19770,7091,11,20260407,Pass,90,Chicago,5,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633624,27432,11346,30,20260407,Fail,70,Chicago,6,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633586,6261,1,30,20260407,Fail,70,Chicago,7,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633604,28171,14750,22,20260407,Pass,90,Chicago,8,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load
2633595,32765,null,15,20260407,Pass,90,Chicago,9,2026-04-13T18:22:51.781Z,2026-04-13T18:22:51.781Z,/Users/ravi.ara@northeastern.edu/FoodInspection_FinalProject/test/04_gold_dim_load


---
## 8. bridge_inspection_violation

In [0]:
# Reload fact and violation dim
fact_inspection = spark.table(f"{GOLD_SCHEMA}.fact_inspection")
dim_violation = spark.table(f"{GOLD_SCHEMA}.dim_violation")

In [0]:
# Chicago bridge
df_chi_bridge = (
    df_chi_viol
    .join(
        fact_inspection.select("inspection_key", "inspection_id", "source_city"),
        (df_chi_viol["Inspection_ID"] == fact_inspection["inspection_id"]) &
        (df_chi_viol["source_city"] == fact_inspection["source_city"]),
        "inner"
    )
    .join(
        dim_violation,
        (coalesce(df_chi_viol["violation_code"], lit("")) == coalesce(dim_violation["violation_code"], lit(""))) &
        (df_chi_viol["violation_description"] == dim_violation["violation_description"]) &
        (df_chi_viol["source_city"] == dim_violation["source_city"]),
        "inner"
    )

    .select(
        fact_inspection["inspection_key"],
        dim_violation["violation_key"],
        df_chi_viol["violation_comments"]
    )
)

# Dallas bridge
df_dal_bridge = (
    df_dal_viol
    .join(
        fact_inspection.select("inspection_key", "inspection_id", "source_city"),
        (df_dal_viol["Inspection_ID"] == fact_inspection["inspection_id"]) &
        (df_dal_viol["source_city"] == fact_inspection["source_city"]),
        "inner"
    )
    .join(
        dim_violation,
        (coalesce(df_dal_viol["violation_code"], lit("")) == coalesce(dim_violation["violation_code"], lit(""))) &
        (df_dal_viol["violation_description"] == dim_violation["violation_description"]) &
        (df_dal_viol["source_city"] == dim_violation["source_city"]),
        "inner"
    )

    .select(
        fact_inspection["inspection_key"],
        dim_violation["violation_key"],
        df_dal_viol["violation_comments"]
    )
)

# Union and write
df_bridge = df_chi_bridge.unionByName(df_dal_bridge)

df_bridge = add_gold_lineage(df_bridge)

(
    df_bridge.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD_SCHEMA}.bridge_inspection_violation")
)

write_parquet_snapshot(df_bridge, GOLD_PATH, "bridge_inspection_violation")

print(f"bridge_inspection_violation: {df_bridge.count()} rows")

Parquet snapshot written to: /Volumes/workspace/foodinspection_gold/food_inspection/bridge_inspection_violation/2026/04/13/run_id=20260413_182145
bridge_inspection_violation: 1263794 rows


---
## 9. Verify Gold Layer

In [0]:
display(spark.sql(f"""
    SELECT 'dim_date' AS table_name, COUNT(*) AS row_count FROM {GOLD_SCHEMA}.dim_date
    UNION ALL SELECT 'dim_restaurant', COUNT(*) FROM {GOLD_SCHEMA}.dim_restaurant
    UNION ALL SELECT 'dim_location', COUNT(*) FROM {GOLD_SCHEMA}.dim_location
    UNION ALL SELECT 'dim_inspection_type', COUNT(*) FROM {GOLD_SCHEMA}.dim_inspection_type
    UNION ALL SELECT 'dim_violation', COUNT(*) FROM {GOLD_SCHEMA}.dim_violation
    UNION ALL SELECT 'fact_inspection', COUNT(*) FROM {GOLD_SCHEMA}.fact_inspection
    UNION ALL SELECT 'bridge_inspection_violation', COUNT(*) FROM {GOLD_SCHEMA}.bridge_inspection_violation
"""))

table_name,row_count
dim_date,5939
dim_restaurant,45544
dim_location,25977
dim_inspection_type,64
dim_violation,15465
fact_inspection,275967
bridge_inspection_violation,1263794
